# LeetCode 1742 — Maximum Number of Balls in a Box bằng Python

Link đề: https://leetcode.com/problems/maximum-number-of-balls-in-a-box/

Độ khó: **Easy** · Chủ đề: Hash Table, Math, Counting

Skeleton LeetCode:

```python
class Solution:
    def countBalls(self, lowLimit: int, highLimit: int) -> int:

```

Mục tiêu học:

1. Tính **tổng chữ số** (digit sum) của một số theo hai cách: qua chuỗi `str`, và qua toán `% 10`, `// 10`.
2. Dùng **dictionary đếm tần suất** (`dict`, `dict.get`, `collections.Counter`) — mẫu "đếm rồi lấy max".
3. Nhận ra miền giá trị nhỏ (tổng chữ số ≤ 45) → thay dict bằng **mảng đếm cố định**.
4. Học mẹo **cập nhật tổng chữ số tăng dần** khi đi từ `n` sang `n + 1` (xử lý các chữ số 9 ở cuối).
5. Phân tích `O(N · d)` so với `O(N)` khấu hao.
6. Tự giải các biến thể: hộp nào nhiều bóng nhất, đếm theo tích chữ số, …

## 1. Bài toán yêu cầu gì?

Có các quả bóng đánh số từ `lowLimit` đến `highLimit` (bao gồm cả hai đầu), và vô số hộp đánh số `1, 2, 3, …`.

Bóng số `x` được bỏ vào **hộp có số = tổng các chữ số của `x`**. Ví dụ bóng `321` → hộp `3+2+1 = 6`; bóng `10` → hộp `1+0 = 1`.

Trả về **số bóng trong hộp chứa nhiều bóng nhất**.

| lowLimit | highLimit | Phân bố đáng chú ý | Kết quả |
|---:|---:|---|---:|
| 1 | 10 | hộp 1 có bóng 1 và 10 | 2 |
| 5 | 15 | hộp 5 có {5, 14}, hộp 6 có {6, 15} | 2 |
| 19 | 28 | hộp 10 có {19, 28} | 2 |

Constraints: `1 <= lowLimit <= highLimit <= 10^5`.

Chú ý: đề hỏi **số lượng bóng**, không hỏi **số hiệu hộp**.

In [1]:
# Nhìn trực quan ví dụ 2: bóng nào vào hộp nào
for ball in range(5, 16):
    box = sum(int(ch) for ch in str(ball))
    print(f"bóng {ball:>2} -> hộp {box}")

bóng  5 -> hộp 5
bóng  6 -> hộp 6
bóng  7 -> hộp 7
bóng  8 -> hộp 8
bóng  9 -> hộp 9
bóng 10 -> hộp 1
bóng 11 -> hộp 2
bóng 12 -> hộp 3
bóng 13 -> hộp 4
bóng 14 -> hộp 5
bóng 15 -> hộp 6


## 2. Nền tảng Python cần biết

### 2.1. Tổng chữ số qua chuỗi

`str(321)` → `"321"`. Duyệt từng ký tự, `int(ch)` đổi `'3'` → `3`, rồi cộng lại.

In [2]:
x = 321
s = str(x)
print(list(s))
print([int(ch) for ch in s])
print(sum(int(ch) for ch in s))

['3', '2', '1']
[3, 2, 1]
6


`[int(ch) for ch in s]` là **list comprehension**: "tạo list gồm `int(ch)` với mỗi `ch` trong `s`". Bỏ dấu `[]` (viết thẳng trong `sum(...)`) thành **generator expression** — không tạo list trung gian.

### 2.2. Tổng chữ số bằng toán: `% 10` và `// 10`

- `x % 10` lấy **chữ số cuối**: `321 % 10 = 1`.
- `x // 10` **bỏ chữ số cuối**: `321 // 10 = 32`.

Lặp lại cho tới khi `x == 0`.

In [3]:
x = 321
print(x % 10, x // 10)

def digit_sum(x: int) -> int:
    total = 0
    while x > 0:
        total += x % 10
        x //= 10
    return total

for v in [321, 10, 99999, 7, 100000]:
    print(v, "->", digit_sum(v))

1 32
321 -> 6
10 -> 1
99999 -> 45
7 -> 7
100000 -> 1


### 2.3. `divmod(x, 10)` — lấy cả thương và dư một lần

`divmod(321, 10)` trả `(32, 1)`. Gán hai biến cùng lúc: `x, d = divmod(x, 10)`.

In [4]:
print(divmod(321, 10))
x, d = divmod(321, 10)
print("x =", x, "| d =", d)

(32, 1)
x = 32 | d = 1


### 2.4. Dictionary đếm tần suất

`dict` lưu cặp **key → value**. Ở bài này: key = số hộp, value = số bóng trong hộp.

`counts.get(box, 0)` đọc giá trị nếu có, nếu chưa có key thì trả `0` (không lỗi).

In [5]:
counts = {}
for box in [1, 2, 1, 3, 1]:
    counts[box] = counts.get(box, 0) + 1
    print("thêm vào hộp", box, "->", counts)
print("hộp 1 có", counts[1], "bóng")
print("hộp 9 có", counts.get(9, 0), "bóng")

thêm vào hộp 1 -> {1: 1}
thêm vào hộp 2 -> {1: 1, 2: 1}
thêm vào hộp 1 -> {1: 2, 2: 1}
thêm vào hộp 3 -> {1: 2, 2: 1, 3: 1}
thêm vào hộp 1 -> {1: 3, 2: 1, 3: 1}
hộp 1 có 3 bóng
hộp 9 có 0 bóng


### 2.5. `collections.Counter`

`Counter(iterable)` đếm tần suất sẵn cho ta. `c.most_common(1)` trả `[(key, số_lần)]` của phần tử xuất hiện nhiều nhất. `max(c.values())` lấy tần suất lớn nhất.

In [6]:
from collections import Counter

c = Counter([1, 2, 1, 3, 1])
print(c)
print(c.most_common(1))
print(max(c.values()))

Counter({1: 3, 2: 1, 3: 1})
[(1, 3)]
3


### 2.6. `range(a, b + 1)` để duyệt **bao gồm** hai đầu

`range(a, b)` dừng **trước** `b`. Đề nói "inclusive" nên phải dùng `range(lowLimit, highLimit + 1)`.

In [7]:
print(list(range(5, 10)), "<- thiếu 10")
print(list(range(5, 10 + 1)))

[5, 6, 7, 8, 9] <- thiếu 10
[5, 6, 7, 8, 9, 10]


### 2.7. Tổng chữ số lớn nhất có thể là bao nhiêu?

Với số ≤ `10^5 = 100000`, số có tổng chữ số lớn nhất là `99999` → `45`. Vậy chỉ có **tối đa 45 hộp** được dùng (hộp 1..45). Đây là chìa khóa để dùng mảng thay cho dict.

In [8]:
print("tổng chữ số lớn nhất trong [1, 100000]:", max(digit_sum(v) for v in range(1, 100001)))

tổng chữ số lớn nhất trong [1, 100000]: 45


## 3. Cách trực giác — mô phỏng: bỏ từng bóng vào hộp

Đề gợi ý: giới hạn nhỏ (≤ 10^5 bóng) nên cứ duyệt hết.

1. `counts = {}`.
2. Với mỗi bóng `x` từ `lowLimit` đến `highLimit`: tính `box = digit_sum(x)`, tăng `counts[box]`.
3. Trả `max(counts.values())`.

In [9]:
def count_balls_dict(lowLimit: int, highLimit: int) -> int:
    counts = {}
    for ball in range(lowLimit, highLimit + 1):
        box = sum(int(ch) for ch in str(ball))
        counts[box] = counts.get(box, 0) + 1
    return max(counts.values())

print(count_balls_dict(1, 10))    # 2
print(count_balls_dict(5, 15))    # 2
print(count_balls_dict(19, 28))   # 2

2
2
2


### 3.1. Một cách "ngây thơ" hơn: thử từng hộp, đếm lại toàn bộ bóng

Có người viết: với mỗi hộp `b` từ 1 đến 45, đếm xem bao nhiêu bóng có tổng chữ số bằng `b`. Đúng, nhưng duyệt danh sách bóng **45 lần**.

In [10]:
def count_balls_per_box(lowLimit: int, highLimit: int) -> int:
    best = 0
    for b in range(1, 46):
        cnt = 0
        for ball in range(lowLimit, highLimit + 1):
            if digit_sum(ball) == b:
                cnt += 1
        best = max(best, cnt)
    return best

print(count_balls_per_box(1, 10), count_balls_per_box(5, 15), count_balls_per_box(19, 28))

2 2 2


## 4. Nút thắt và insight

**Nút thắt 1 (cách thử từng hộp):** mỗi bóng bị tính tổng chữ số 45 lần. Ta nên **đi qua bóng một lần**, bỏ ngay vào hộp của nó — đó là đếm tần suất.

**Nút thắt 2 (dict):** dict linh hoạt nhưng có chi phí băm. Vì số hộp ≤ 45, một **list dài 46** (`counts[box]`) là đủ — truy cập theo chỉ số trực tiếp.

**Nút thắt 3 (tổng chữ số):** mỗi bóng tốn `d` phép chia (d = số chữ số ≤ 6). Có thể làm tốt hơn: tổng chữ số của `n + 1` suy ra từ tổng chữ số của `n`:

- Nếu `n` **không** tận cùng bằng 9: `S(n+1) = S(n) + 1`. Ví dụ `S(23) = 5` → `S(24) = 6`.
- Nếu `n` tận cùng bằng `k` chữ số 9: các số 9 đó thành 0 (mất `9k`) và chữ số kế trước tăng 1. `S(n+1) = S(n) + 1 - 9k`. Ví dụ `S(199) = 19` → `S(200) = 19 + 1 - 18 = 2`.

In [11]:
for n in [23, 29, 199, 999, 1099]:
    k, m = 0, n
    while m % 10 == 9:
        k += 1
        m //= 10
    print(f"S({n}) = {digit_sum(n):>2}, số 9 cuối k = {k} -> S({n+1}) = {digit_sum(n)} + 1 - 9*{k} = {digit_sum(n) + 1 - 9*k} (thật: {digit_sum(n+1)})")

S(23) =  5, số 9 cuối k = 0 -> S(24) = 5 + 1 - 9*0 = 6 (thật: 6)
S(29) = 11, số 9 cuối k = 1 -> S(30) = 11 + 1 - 9*1 = 3 (thật: 3)
S(199) = 19, số 9 cuối k = 2 -> S(200) = 19 + 1 - 9*2 = 2 (thật: 2)
S(999) = 27, số 9 cuối k = 3 -> S(1000) = 27 + 1 - 9*3 = 1 (thật: 1)
S(1099) = 19, số 9 cuối k = 2 -> S(1100) = 19 + 1 - 9*2 = 2 (thật: 2)


## 5. Lời giải chính — mảng đếm cố định 46 ô

### 5.1. Tạo mảng đếm

`[0] * 46` tạo list 46 số 0, chỉ số `0..45`. Ta dùng chỉ số = số hộp (bỏ trống ô 0, vì bóng ≥ 1 nên tổng chữ số ≥ 1).

In [12]:
counts = [0] * 46
print(len(counts), counts[:10])

46 [0, 0, 0, 0, 0, 0, 0, 0, 0, 0]


### 5.2. Bỏ một bóng vào hộp

In [13]:
counts = [0] * 46
for ball in [19, 28, 20]:
    box = digit_sum(ball)
    counts[box] += 1
    print(f"bóng {ball} -> hộp {box}; counts[{box}] = {counts[box]}")

bóng 19 -> hộp 10; counts[10] = 1
bóng 28 -> hộp 10; counts[10] = 2
bóng 20 -> hộp 2; counts[2] = 1


### 5.3. Ghép lại

In [14]:
def count_balls(lowLimit: int, highLimit: int) -> int:
    counts = [0] * 46                  # hộp 1..45 là đủ cho số <= 10^5
    for ball in range(lowLimit, highLimit + 1):
        box = 0
        x = ball
        while x > 0:                   # tổng chữ số bằng toán
            box += x % 10
            x //= 10
        counts[box] += 1
    return max(counts)

print(count_balls(1, 10), count_balls(5, 15), count_balls(19, 28))
print(count_balls(1, 100000))

2 2 2
6000


## 6. TRACE từng bước

Bảng trace cho `lowLimit = 19`, `highLimit = 28` (chỉ ghi các hộp khác 0):

| ball | tổng chữ số | hộp được cộng | trạng thái các hộp sau bước |
|---:|---:|---:|---|
| 19 | 1+9=10 | 10 | {10:1} |
| 20 | 2+0=2 | 2 | {2:1, 10:1} |
| 21 | 3 | 3 | {2:1, 3:1, 10:1} |
| … | … | … | … |
| 27 | 9 | 9 | {2..9: 1, 10:1} |
| 28 | 2+8=10 | 10 | {2..9: 1, **10:2**} |

Hộp 10 nhận 2 bóng (19 và 28) → đáp án 2.

In [15]:
def trace_count_balls(lowLimit: int, highLimit: int) -> int:
    counts = [0] * 46
    print(f"{'ball':>5} | {'các chữ số':<12} | box | counts[box] | max hiện tại")
    best = 0
    for ball in range(lowLimit, highLimit + 1):
        digits = [int(ch) for ch in str(ball)]
        box = sum(digits)
        counts[box] += 1
        best = max(best, counts[box])
        print(f"{ball:>5} | {str(digits):<12} | {box:>3} | {counts[box]:>10} | {best}")
    nonzero = {b: c for b, c in enumerate(counts) if c}
    print("Các hộp có bóng:", nonzero)
    print("=> trả về", max(counts))
    return max(counts)

trace_count_balls(19, 28)

 ball | các chữ số   | box | counts[box] | max hiện tại
   19 | [1, 9]       |  10 |          1 | 1
   20 | [2, 0]       |   2 |          1 | 1
   21 | [2, 1]       |   3 |          1 | 1
   22 | [2, 2]       |   4 |          1 | 1
   23 | [2, 3]       |   5 |          1 | 1
   24 | [2, 4]       |   6 |          1 | 1
   25 | [2, 5]       |   7 |          1 | 1
   26 | [2, 6]       |   8 |          1 | 1
   27 | [2, 7]       |   9 |          1 | 1
   28 | [2, 8]       |  10 |          2 | 2
Các hộp có bóng: {2: 1, 3: 1, 4: 1, 5: 1, 6: 1, 7: 1, 8: 1, 9: 1, 10: 2}
=> trả về 2


2

### 6.1. Trace ví dụ 2 — hai hộp hòa nhau

Hộp 5 và hộp 6 cùng có 2 bóng. Đề chỉ hỏi số lượng → vẫn trả 2.

In [16]:
trace_count_balls(5, 15)

 ball | các chữ số   | box | counts[box] | max hiện tại
    5 | [5]          |   5 |          1 | 1
    6 | [6]          |   6 |          1 | 1
    7 | [7]          |   7 |          1 | 1
    8 | [8]          |   8 |          1 | 1
    9 | [9]          |   9 |          1 | 1
   10 | [1, 0]       |   1 |          1 | 1
   11 | [1, 1]       |   2 |          1 | 1
   12 | [1, 2]       |   3 |          1 | 1
   13 | [1, 3]       |   4 |          1 | 1
   14 | [1, 4]       |   5 |          2 | 2
   15 | [1, 5]       |   6 |          2 | 2
Các hộp có bóng: {1: 1, 2: 1, 3: 1, 4: 1, 5: 2, 6: 2, 7: 1, 8: 1, 9: 1}
=> trả về 2


2

`{b: c for b, c in enumerate(counts) if c}` là **dict comprehension**: tạo dict `số_hộp → số_bóng` chỉ cho các hộp khác 0 (`if c` đúng khi `c != 0`).

## 7. Code nộp LeetCode

Bản gọn, không in debug, đúng chữ ký `countBalls(self, lowLimit, highLimit)`.

In [17]:
class Solution:
    def countBalls(self, lowLimit: int, highLimit: int) -> int:
        counts = [0] * 46
        for ball in range(lowLimit, highLimit + 1):
            box = 0
            x = ball
            while x:
                box += x % 10
                x //= 10
            counts[box] += 1
        return max(counts)

sol = Solution()
print(sol.countBalls(1, 10), sol.countBalls(5, 15), sol.countBalls(19, 28))

2 2 2


## 8. Các lời giải khác

### 8.1. `Counter` + chuỗi — ngắn gọn nhất

In [18]:
from collections import Counter

class SolutionCounter:
    def countBalls(self, lowLimit: int, highLimit: int) -> int:
        boxes = Counter(sum(map(int, str(b))) for b in range(lowLimit, highLimit + 1))
        return max(boxes.values())

print(SolutionCounter().countBalls(19, 28))

2


`map(int, "321")` áp `int` lên từng ký tự → `3, 2, 1`. `sum(map(int, str(b)))` là tổng chữ số viết một dòng.

In [19]:
print(list(map(int, "321")), sum(map(int, str(321))))

[3, 2, 1] 6


### 8.2. Cập nhật tổng chữ số tăng dần — `O(1)` khấu hao mỗi bóng

Tính `S(lowLimit)` một lần. Sau đó với mỗi bóng tiếp theo, dùng công thức `S(n+1) = S(n) + 1 - 9k` (k = số chữ số 9 ở cuối `n`).

Vì sao nhanh? Chỉ 1/10 số tận cùng bằng 9, 1/100 tận cùng bằng 99, … → trung bình số vòng `while` mỗi bóng ≈ `1/10 + 1/100 + … < 0.12`.

In [20]:
def count_balls_incremental(lowLimit: int, highLimit: int) -> int:
    counts = [0] * 46
    s = digit_sum(lowLimit)
    for n in range(lowLimit, highLimit + 1):
        counts[s] += 1
        # chuẩn bị S(n + 1)
        m = n
        while m % 10 == 9:       # mỗi số 9 cuối biến thành 0
            s -= 9
            m //= 10
        s += 1                   # chữ số kế tiếp tăng 1
    return max(counts)

print(count_balls_incremental(1, 10), count_balls_incremental(5, 15), count_balls_incremental(19, 28))
print(count_balls_incremental(1, 100000))

2 2 2
6000


In [21]:
class SolutionIncremental:
    def countBalls(self, lowLimit: int, highLimit: int) -> int:
        counts = [0] * 46
        x, s = lowLimit, 0
        while x:
            s += x % 10
            x //= 10
        for n in range(lowLimit, highLimit + 1):
            counts[s] += 1
            m = n
            while m % 10 == 9:
                s -= 9
                m //= 10
            s += 1
        return max(counts)

print(SolutionIncremental().countBalls(5, 15))

2


### 8.3. Đo nhanh thời gian (minh họa, không phải benchmark chính xác)

`time.perf_counter()` đọc đồng hồ độ phân giải cao; hiệu hai lần đọc = thời gian chạy (giây). Con số phụ thuộc máy, chỉ dùng để so tương đối.

In [22]:
import time

for name, fn in [("dict + str", count_balls_dict),
                 ("mảng + toán", count_balls),
                 ("tăng dần", count_balls_incremental)]:
    t0 = time.perf_counter()
    r = fn(1, 100000)
    dt = time.perf_counter() - t0
    print(f"{name:<12} kết quả={r}  ~{dt*1000:.0f} ms")

dict + str   kết quả=6000  ~104 ms


mảng + toán  kết quả=6000  ~27 ms
tăng dần     kết quả=6000  ~8 ms


### 8.4. Bảng so sánh

Gọi `N = highLimit - lowLimit + 1` (số bóng), `d` = số chữ số (≤ 6), `B` = số hộp (≤ 45).

| Cách | Ý tưởng | Time | Space | Ghi chú |
|---|---|---:|---:|---|
| Thử từng hộp | 45 lần duyệt toàn bộ bóng | `O(B · N · d)` | `O(1)` | Chậm nhất, để thấy nút thắt |
| dict / Counter + `str` | đếm tần suất | `O(N · d)` | `O(B)` | Ngắn, dễ đọc |
| Mảng 46 ô + `%10` | đếm bằng chỉ số | `O(N · d)` | `O(1)` | **Lời giải chính** |
| Tăng dần `S(n+1)` | xử lý số 9 cuối | `O(N + d)` khấu hao | `O(1)` | Sáng tạo, nhanh nhất |

## 9. Vì sao lời giải đúng?

**Bất biến:** sau khi xử lý các bóng `lowLimit … x`, `counts[b]` = số bóng trong đoạn đó có tổng chữ số bằng `b`.

- **Khởi đầu:** chưa có bóng nào, mọi `counts[b] = 0`. Đúng.
- **Duy trì:** bóng `x` có đúng một hộp `digit_sum(x)` → chỉ ô đó tăng 1. Đúng.
- **Kết thúc:** mọi bóng đã vào đúng hộp; `max(counts)` = số bóng nhiều nhất trong một hộp.

**Mảng 46 ô đủ chỗ:** mọi bóng ≤ `10^5` có tổng chữ số ≤ 45 (số `100000` có tổng 1; số `99999` có tổng 45). Không bao giờ truy cập ngoài chỉ số.

**Công thức tăng dần đúng:** cộng 1 vào `n` làm mọi chữ số 9 ở cuối thành 0 (mỗi cái giảm 9) và chữ số đầu tiên khác 9 (từ phải) tăng 1 — đúng phép "nhớ" khi cộng.

In [23]:
# Kiểm chứng công thức tăng dần trên mọi n trong [1, 100000)
s = digit_sum(1)
for n in range(1, 100000):
    m, nxt = n, s
    while m % 10 == 9:
        nxt -= 9
        m //= 10
    nxt += 1
    assert nxt == digit_sum(n + 1), n
    s = nxt
print("S(n+1) = S(n) + 1 - 9k đúng cho mọi n < 100000")

S(n+1) = S(n) + 1 - 9k đúng cho mọi n < 100000


## 10. Độ phức tạp

Gọi `N = highLimit - lowLimit + 1`, `d = số chữ số của highLimit` (≤ 6).

- **Lời giải chính:** mỗi bóng tốn `O(d)` để tính tổng chữ số → **Time `O(N · d)`**. Mảng 46 ô là kích thước cố định → **Space `O(1)`**.
- **Tăng dần:** tổng số vòng xử lý số 9 trên cả đoạn ≤ `N/10 + N/100 + … < N/9` → **Time `O(N + d)`**, **Space `O(1)`**.
- Dùng `dict`: Space `O(B)` với `B ≤ 45` → cũng có thể coi là `O(1)`.

Với `N ≤ 10^5`, mọi cách (trừ "thử từng hộp") đều chạy nhanh.

## 11. Bộ test

Nhóm test:

1. Ba ví dụ của đề.
2. Một bóng (`low == high`).
3. Đoạn chứa "bước nhảy" tổng chữ số: `9 → 10`, `99 → 100`, `99999 → 100000`.
4. Đoạn lớn nhất `[1, 100000]`.
5. Đối chiếu mọi cách trên nhiều đoạn ngẫu nhiên (seed cố định).

In [24]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"{name:<26} PASS (={actual})")

sol = Solution()
check("đề 1", sol.countBalls(1, 10), 2)
check("đề 2", sol.countBalls(5, 15), 2)
check("đề 3", sol.countBalls(19, 28), 2)
check("một bóng", sol.countBalls(7, 7), 1)
check("1..9 mỗi hộp 1 bóng", sol.countBalls(1, 9), 1)
check("qua 99 -> 100", sol.countBalls(99, 100), 1)
check("bóng 100000", sol.countBalls(100000, 100000), 1)
check("toàn miền", sol.countBalls(1, 100000), count_balls_dict(1, 100000))

đề 1                       PASS (=2)
đề 2                       PASS (=2)
đề 3                       PASS (=2)
một bóng                   PASS (=1)
1..9 mỗi hộp 1 bóng        PASS (=1)
qua 99 -> 100              PASS (=1)
bóng 100000                PASS (=1)
toàn miền                  PASS (=6000)


In [25]:
import random

rng = random.Random(1742)
impls = [count_balls_dict, count_balls, count_balls_incremental,
         lambda a, b: SolutionCounter().countBalls(a, b),
         lambda a, b: SolutionIncremental().countBalls(a, b)]
for _ in range(200):
    a = rng.randint(1, 100000)
    b = min(100000, a + rng.randint(0, 3000))
    outs = {f(a, b) for f in impls}
    assert len(outs) == 1, (a, b, outs)
# thêm vài đoạn nhỏ so với cách "thử từng hộp"
for a, b in [(1, 50), (95, 130), (990, 1010)]:
    assert count_balls_per_box(a, b) == count_balls(a, b)
print("200 đoạn ngẫu nhiên + 3 đoạn nhỏ: mọi cách cho cùng kết quả")

200 đoạn ngẫu nhiên + 3 đoạn nhỏ: mọi cách cho cùng kết quả


## 12. Lỗi người mới hay gặp

### Lỗi 1: `range(lowLimit, highLimit)` — bỏ sót bóng cuối

In [26]:
def wrong_range(lo, hi):
    counts = [0] * 46
    for ball in range(lo, hi):          # thiếu hi
        counts[digit_sum(ball)] += 1
    return max(counts)

print("sai:", wrong_range(19, 28), "| đúng:", count_balls(19, 28))

sai: 1 | đúng: 2


**Sửa:** `range(lowLimit, highLimit + 1)`.

### Lỗi 2: Trả về **số hộp** thay vì **số bóng**

In [27]:
def wrong_return_box(lo, hi):
    counts = [0] * 46
    for ball in range(lo, hi + 1):
        counts[digit_sum(ball)] += 1
    return counts.index(max(counts))    # đây là SỐ HỘP

print("sai:", wrong_return_box(19, 28), "| đúng:", count_balls(19, 28))

sai: 10 | đúng: 2


### Lỗi 3: Mảng đếm quá nhỏ

Ví dụ `[0] * 45` (chỉ số 0..44) → bóng `99999` có tổng 45 gây `IndexError`.

In [28]:
counts = [0] * 45
try:
    counts[digit_sum(99999)] += 1
except IndexError as e:
    print("IndexError:", e)

IndexError: list index out of range


**Sửa:** `[0] * 46`, hoặc dùng dict cho an toàn.

### Lỗi 4: Phá hỏng biến vòng lặp khi tính tổng chữ số

Viết `while ball > 0: ball //= 10` ngay trên biến vòng `for`. Trong Python vòng `for` vẫn chạy đúng (vì `range` cấp giá trị mới mỗi vòng), nhưng nếu sau đó còn dùng `ball` (ví dụ để in, hoặc ở ngôn ngữ khác dùng `while`) sẽ sai. **Sửa:** sao chép sang biến tạm `x = ball`.

In [29]:
for ball in range(19, 22):
    tmp_sum = 0
    while ball > 0:
        tmp_sum += ball % 10
        ball //= 10
    print("tổng =", tmp_sum, "| nhưng ball giờ là", ball, "<- mất số bóng gốc")

tổng = 10 | nhưng ball giờ là 0 <- mất số bóng gốc
tổng = 2 | nhưng ball giờ là 0 <- mất số bóng gốc
tổng = 3 | nhưng ball giờ là 0 <- mất số bóng gốc


### Lỗi 5: Dùng `/ 10` thay vì `// 10`

`/` cho số thực → vòng `while x > 0` chạy rất lâu và tổng sai. **Sửa:** `//`.

In [30]:
print(321 / 10, 321 // 10)

32.1 32


## 13. Mẹo nhận dạng pattern

- **"Mỗi phần tử thuộc về một nhóm theo một khóa tính được"** + hỏi "nhóm lớn nhất" → **đếm tần suất** (dict / Counter / mảng đếm), rồi `max`.
- **Miền khóa nhỏ và biết trước** (ở đây 1..45) → **mảng đếm** thay dict.
- **Tổng chữ số** → `% 10` / `// 10`, hoặc `sum(map(int, str(x)))`.
- Duyệt các số **liên tiếp** và cần một đại lượng theo chữ số → nghĩ tới **cập nhật tăng dần** thay vì tính lại từ đầu.
- Giới hạn `≤ 10^5` → mô phỏng trực tiếp thường là đủ.

Bài cùng họ: 1399 Count Largest Group, 2342 Max Sum of a Pair With Equal Sum of Digits, 258 Add Digits, 1281 Subtract the Product and Sum of Digits.

## 14. Biến thể

### 14.1. Trả về **số hiệu hộp** nhiều bóng nhất (hộp nhỏ nhất nếu hòa)

`counts.index(max(counts))` trả chỉ số **đầu tiên** đạt max → đúng hộp nhỏ nhất.

In [31]:
def busiest_box(lo, hi):
    counts = [0] * 46
    for ball in range(lo, hi + 1):
        counts[digit_sum(ball)] += 1
    return counts.index(max(counts))

print(busiest_box(5, 15))   # hộp 5 và 6 cùng 2 bóng -> 5
print(busiest_box(1, 10))   # 1

5
1


### 14.2. Bài 1399 — Đếm số nhóm có kích thước lớn nhất

Nhóm các số `1..n` theo tổng chữ số; đếm có bao nhiêu nhóm đạt kích thước lớn nhất.

In [32]:
def count_largest_group(n):
    counts = Counter(digit_sum(x) for x in range(1, n + 1))
    best = max(counts.values())
    return sum(1 for v in counts.values() if v == best)

print(count_largest_group(13))  # nhóm [1,10],[2,11],[3,12],[4,13] -> 4
print(count_largest_group(2))   # 2

4
2


### 14.3. Phân hộp theo **tích** chữ số

Đổi "khóa" từ tổng sang tích — khung đếm tần suất giữ nguyên, chỉ cần dict vì miền khóa lớn hơn.

In [33]:
def digit_product(x):
    p = 1
    while x:
        p *= x % 10
        x //= 10
    return p

def max_balls_by_product(lo, hi):
    return max(Counter(digit_product(b) for b in range(lo, hi + 1)).values())

print(max_balls_by_product(1, 30))   # tích 0: {10, 20, 30} -> 3

3


## 15. Bài tập tự luyện

Các hàm TODO không được gọi khi chạy notebook. Tự làm trước, đáp án ở mục 17.

### Bài 1 — Tính tay

Với `lowLimit = 10`, `highLimit = 20`, liệt kê hộp của từng bóng và cho đáp án.

### Bài 2 — Dự đoán output

`digit_sum(1009)`, `sum(map(int, "1009"))`, `divmod(1009, 10)` lần lượt ra gì?

### Bài 3 — Điền TODO: hàm tổng chữ số bằng toán

In [34]:
def exercise_digit_sum(x: int) -> int:
    # TODO 1: total = 0
    # TODO 2: while x > 0: cộng x % 10, rồi x //= 10
    pass

# assert exercise_digit_sum(321) == 6
# assert exercise_digit_sum(100000) == 1

### Bài 4 — Tự viết lời giải bằng dict

In [35]:
def exercise_count_balls(lowLimit: int, highLimit: int) -> int:
    # TODO: counts = {}; duyệt bóng (nhớ +1); counts.get(box, 0) + 1; trả max(values)
    pass

# assert exercise_count_balls(19, 28) == 2

### Bài 5 — Biến thể: liệt kê **tất cả** hộp đạt max (tăng dần)

In [36]:
def exercise_all_busiest_boxes(lowLimit: int, highLimit: int) -> list[int]:
    # TODO: đếm, tìm max, lọc các hộp có số bóng == max
    pass

# assert exercise_all_busiest_boxes(5, 15) == [5, 6]
# assert exercise_all_busiest_boxes(1, 10) == [1]

### Bài 6 — Mini challenge: bài 2342 rút gọn

Cho list `nums`, tìm `nums[i] + nums[j]` lớn nhất (i ≠ j) sao cho hai số có **cùng tổng chữ số**. Không có cặp → `-1`. Yêu cầu một lần duyệt, dùng dict lưu "số lớn nhất đã gặp" của mỗi tổng chữ số.

In [37]:
def exercise_max_pair_equal_digit_sum(nums: list[int]) -> int:
    # TODO: best_by_sum = {}; với mỗi x: s = digit_sum(x); nếu s đã có -> cập nhật đáp án
    pass

# assert exercise_max_pair_equal_digit_sum([18, 43, 36, 13, 7]) == 54   # 18 + 36
# assert exercise_max_pair_equal_digit_sum([10, 12, 19, 14]) == -1

## 16. Gợi ý 3 mức

**Bài 1** — M1: tính tổng chữ số từng số 10..20. M2: 10→1, 11→2, …, 19→10, 20→2. M3: hộp 2 có {11, 20}.

**Bài 2** — M1: 1+0+0+9. M2: `divmod` trả (thương, dư). M3: `10`, `10`, `(100, 9)`.

**Bài 3** — M1: lấy chữ số cuối bằng `%`. M2: bỏ chữ số cuối bằng `//`. M3: `total += x % 10; x //= 10`.

**Bài 4** — M1: `range(lowLimit, highLimit + 1)`. M2: `box = exercise_digit_sum(ball)` hoặc `sum(map(int, str(ball)))`. M3: `return max(counts.values())`.

**Bài 5** — M1: tìm `best = max(...)` trước. M2: duyệt `sorted(counts)` (theo số hộp). M3: `[b for b in sorted(counts) if counts[b] == best]`.

**Bài 6** — M1: với mỗi tổng chữ số chỉ cần nhớ **số lớn nhất** đã gặp. M2: gặp `x` có tổng `s` đã có trong dict → ứng viên `x + best_by_sum[s]`. M3: sau đó `best_by_sum[s] = max(best_by_sum[s], x)`.

## 17. Đáp án tham khảo

**Bài 1:** hộp của 10..20 là `1,2,3,4,5,6,7,8,9,10,2` → hộp 2 có 2 bóng (11 và 20) → đáp án **2**.

**Bài 2:** `10`, `10`, `(100, 9)`.

In [38]:
print([digit_sum(b) for b in range(10, 21)], count_balls(10, 20))
print(digit_sum(1009), sum(map(int, "1009")), divmod(1009, 10))

[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 2] 2
10 10 (100, 9)


In [39]:
def answer_digit_sum(x):
    total = 0
    while x > 0:
        total += x % 10
        x //= 10
    return total


def answer_count_balls(lowLimit, highLimit):
    counts = {}
    for ball in range(lowLimit, highLimit + 1):
        box = answer_digit_sum(ball)
        counts[box] = counts.get(box, 0) + 1
    return max(counts.values())


def answer_all_busiest_boxes(lowLimit, highLimit):
    counts = Counter(answer_digit_sum(b) for b in range(lowLimit, highLimit + 1))
    best = max(counts.values())
    return [b for b in sorted(counts) if counts[b] == best]


def answer_max_pair_equal_digit_sum(nums):
    best_by_sum = {}
    ans = -1
    for x in nums:
        s = answer_digit_sum(x)
        if s in best_by_sum:
            ans = max(ans, x + best_by_sum[s])
            best_by_sum[s] = max(best_by_sum[s], x)
        else:
            best_by_sum[s] = x
    return ans


assert answer_digit_sum(321) == 6
assert answer_digit_sum(100000) == 1
assert answer_count_balls(19, 28) == 2
assert answer_count_balls(1, 100000) == count_balls(1, 100000)
assert answer_all_busiest_boxes(5, 15) == [5, 6]
assert answer_all_busiest_boxes(1, 10) == [1]
assert answer_max_pair_equal_digit_sum([18, 43, 36, 13, 7]) == 54
assert answer_max_pair_equal_digit_sum([10, 12, 19, 14]) == -1

# đối chiếu bài 6 với brute force O(n^2)
rng = random.Random(6)
for _ in range(300):
    nums = [rng.randint(1, 200) for _ in range(rng.randint(1, 10))]
    brute = max([nums[i] + nums[j] for i in range(len(nums)) for j in range(i + 1, len(nums))
                 if digit_sum(nums[i]) == digit_sum(nums[j])], default=-1)
    assert answer_max_pair_equal_digit_sum(nums) == brute
print("Đáp án tham khảo PASS toàn bộ test!")

Đáp án tham khảo PASS toàn bộ test!


## 18. Template nhớ nhanh + checklist submit

```python
class Solution:
    def countBalls(self, lowLimit: int, highLimit: int) -> int:
        counts = [0] * 46                       # tổng chữ số <= 45
        for ball in range(lowLimit, highLimit + 1):   # inclusive!
            box, x = 0, ball
            while x:
                box += x % 10
                x //= 10
            counts[box] += 1
        return max(counts)                      # số bóng, không phải số hộp
```

Template "nhóm theo khóa rồi lấy nhóm lớn nhất":

```text
counts = {}  (hoặc mảng nếu miền khóa nhỏ)
for x in dữ_liệu:
    k = key(x)
    counts[k] += 1
return max(counts.values())
```

Checklist trước khi submit:

- [ ] `range(lowLimit, highLimit + 1)` — có `+ 1`?
- [ ] Mảng đếm đủ lớn (46 ô) hoặc dùng dict?
- [ ] Dùng `// 10`, không phải `/ 10`?
- [ ] Không làm hỏng biến bóng khi tính tổng chữ số?
- [ ] Trả **số bóng** (`max(counts)`), không phải chỉ số hộp?